# SSB ingest — four tables from Statistikkbanken

One notebook, four tables, the same five steps each time:

**search → describe → values → fetch → plot**

Nothing in between. `ssb.fetch()` hands back tidy, typed data — a label column
and a `_code` sibling for every dimension, `value` as a nullable numeric, and
`year` / `date` already parsed — so there is no renaming, splitting or
retyping to do before `storage.save()`.

| SSB table | saved as | drawn by |
|---|---|---|
| 12292 — Omsorgstjenester, supplerende grunnlagstall | `omsorg_bistandsbehov` | `04_pages/03_omsorgstjenester.py` |
| 05810 — Aldersgrupper og kjønnsfordeling | `befolkning_aldersgrupper` | `04_pages/04_befolkning.py` |
| 14657 — Døde per måned | `dode_per_maaned` | `04_pages/05_dodelighet_maaned.py` |
| 07902 — Dødelighetstabeller | `forventet_gjenstaende_levetid` | `04_pages/06_forventet_levetid.py` |

`make notebooks` runs this for its `storage.save()` side effects, so it needs
network — SSB's live API is the source. If SSB is unreachable the pipeline
fails here rather than leaving the pages on stale tables.

In [ ]:
import sys

sys.path.insert(0, "..")  # the notebook runs from 03_notebooks/

import pandas as pd
import plotly.express as px

from backend import storage
from ssb_statistikkbanken import SSB, klass

ssb = SSB()

---
## 12292 — Omsorgstjenester, supplerende grunnlagstall (2015–2025)

### search — which table?

In [ ]:
ssb.search("omsorgstjenester supplerende grunnlagstall", limit=5)

### describe — what is in it?

Three dimensions, all mandatory: region, statistikkvariabel and år. Renders as
a card in Jupyter.

In [ ]:
omsorg_info = ssb.describe("12292")
omsorg_info

### values — which codes can I filter on?

49 statistikkvariabler; `search=` narrows the list instead of eyeballing it.
Four of the hits are the hjemmetjeneste bands this dashboard slices between.

In [ ]:
ssb.values("12292", "ContentsCode", search="omfattende bistandsbehov")

In [ ]:
ssb.values("12292", "KOKkommuneregion0000", search="landet")

### fetch

All four age bands, every region, every published year.

The bands **overlap** — "67 år og over" already contains both "67-79 år" and
"80 år og over" — so they are alternatives to switch between, never a stack to
add up. The page draws them as separate lines for that reason.

In [ ]:
BISTANDSBEHOV = {
    "KOShjelptkjerne00000": "0-66 år",
    "KOShjelptkjerne60001": "67-79 år",
    "KOShjelptkjerne60000": "67 år og over",
    "KOShjelptkjerne80000": "80 år og over",
}

omsorg = ssb.fetch(
    "12292",
    KOKkommuneregion0000="*",   # kommuner, fylker, KOSTRA-grupper — and EAK = Landet
    contents=list(BISTANDSBEHOV),
    time="*",                   # 2015–2025, whatever is published
)

omsorg["aldersgruppe"] = omsorg["ContentsCode_code"].map(BISTANDSBEHOV)
omsorg.shape

### KLASS — keep only the regions that exist today

`KOKkommuneregion0000` mixes 891 values into one dimension: today's kommuner,
every kommune and fylke that has since been merged or split away
(`Halden (-2019)`, `Viken (2020-2023)`), and KOSTRA's municipality groups.

KLASS knows which codes are valid on a given date, so "operational today" is a
join against the classification rather than a hand-kept list that rots at the
next reform. SSB prefixes fylke codes with `EKA`; KLASS uses the bare two
digits, which is the only adjustment needed.

In [ ]:
kommuner = klass.kommuner(2025)
fylker = klass.fylker(2025)

fylke_navn = dict(zip("EKA" + fylker["code"], fylker["name"]))
kommune_til_fylke = klass.correspondence(131, 104, date=2025).set_index("sourceCode")["targetName"]

region_code = omsorg["KOKkommuneregion0000_code"]
omsorg["region_level"] = pd.NA
omsorg.loc[region_code == "EAK", "region_level"] = "Landet"
omsorg.loc[region_code.isin(fylke_navn), "region_level"] = "Fylke"
omsorg.loc[region_code.isin(set(kommuner["code"])), "region_level"] = "Kommune"

# Each kommune carries its fylke, so the page can narrow 357 of them to one county.
omsorg["fylke"] = region_code.map(fylke_navn).fillna(region_code.map(kommune_til_fylke))

omsorg["region_level"].value_counts(dropna=False)

### save

Historic kommuner, historic fylker and the KOSTRA groups drop out here.

A region only reports from the year its current boundaries took effect, so the
empty cells before that go too — the 2020 and 2024 reforms are why the early
years are thin, and SSB does not backcast KOSTRA figures onto new boundaries.

Sorting by band before saving makes SSB's band order the file's row order,
which is where the page reads its category order from.

In [ ]:
rekkefolge = {navn: i for i, navn in enumerate(BISTANDSBEHOV.values())}

omsorg_naa = (
    omsorg[omsorg["region_level"].notna()]
    .dropna(subset=["value"])
    .sort_values(
        ["aldersgruppe", "KOKkommuneregion0000_code", "year"],
        key=lambda col: col.map(rekkefolge) if col.name == "aldersgruppe" else col,
    )
    .reset_index(drop=True)
)

storage.save("omsorg_bistandsbehov", omsorg_naa)
omsorg_naa.head()

How many regions actually report, by year — the shape of the reform gap:

In [ ]:
omsorg_naa.pivot_table(index="year", columns="region_level", values="value", aggfunc="size")

### plot

All four bands for Landet, every available year. `67 år og over` is the two
bands beside it added together, which is why it tracks above both.

In [ ]:
landet = omsorg_naa[omsorg_naa["KOKkommuneregion0000_code"] == "EAK"].sort_values("year")

px.line(
    landet,
    x="year",
    y="value",
    color="aldersgruppe",
    category_orders={"aldersgruppe": list(BISTANDSBEHOV.values())},
    markers=True,
    title="Hjemmetjenestebrukere med omfattende bistandsbehov — Landet",
    labels={"year": "År", "value": "Brukere", "aldersgruppe": "Aldersgruppe"},
)

---
## 05810 — Aldersgrupper og kjønnsfordeling i hele befolkningen (1845–2026)

### search

In [ ]:
ssb.search("aldersgrupper og kjønnsfordeling", limit=5)

### describe

Only `ContentsCode` and `Tid` are mandatory. Kjonn and Alder are eliminable —
leave them out and SSB aggregates them away; pass `"*"` to keep the breakdown.

In [ ]:
befolkning_info = ssb.describe("05810")
befolkning_info

### values

Seven age bands, `999B` being the "Alle" total. `info["Alder"].labels` gives
SSB's own ordering, which is what the chart below uses — alphabetical sorting
would put `7-15 år` after `67-79 år`.

In [ ]:
ssb.values("05810", "Alder")

In [ ]:
ALDER_ORDER = befolkning_info["Alder"].labels
ALDER_ORDER

### fetch

Every sex, every age band, the full 1845–2026 series.

In [ ]:
befolkning = ssb.fetch("05810", Kjonn="*", Alder="*", time="*")

storage.save("befolkning_aldersgrupper", befolkning)
befolkning.head()

### plot

In [ ]:
grupper = befolkning[
    (befolkning["Kjonn_code"] == "0") & (befolkning["Alder_code"] != "999B")
].sort_values("year")

px.area(
    grupper,
    x="year",
    y="value",
    color="Alder",
    category_orders={"Alder": ALDER_ORDER},
    title="Befolkningen etter aldersgruppe, begge kjønn",
    labels={"year": "År", "value": "Personer", "Alder": "Aldersgruppe"},
)

---
## 14657 — Døde per måned, etter kjønn og alder (2022M01–)

### search

In [ ]:
ssb.search("døde per måned", limit=5)

### describe

A monthly table, so `fetch()` returns a parsed `date` column alongside `Tid`.

In [ ]:
dode_info = ssb.describe("14657")
dode_info

### values

`Alder` holds 106 single years — too fine to plot. `variables()` lists the
aggregations SSB publishes for it; `agg_TiAarigGruppering` is ten-year bands.

In [ ]:
ssb.variables("14657")

In [ ]:
ssb.code_list("agg_TiAarigGruppering")

### fetch

`code_list=` selects the aggregation *and* the variable it belongs to, so the
106 single years arrive already rolled up into eleven bands. That is the
"no manual cleanup" part: no binning code here.

In [ ]:
dode = ssb.fetch(
    "14657",
    Kjonn="*",
    code_list={"Alder": "agg_TiAarigGruppering"},
    time="*",
)

storage.save("dode_per_maaned", dode)
dode.head()

### plot

In [ ]:
ALDER_BAND_ORDER = list(dode["Alder"].drop_duplicates())

px.line(
    dode[dode["Kjonn_code"] == "0"].sort_values("date"),
    x="date",
    y="value",
    color="Alder",
    category_orders={"Alder": ALDER_BAND_ORDER},
    title="Døde per måned etter aldersgruppe, begge kjønn",
    labels={"date": "Måned", "value": "Døde", "Alder": "Aldersgruppe"},
)

---
## 07902 — Dødelighetstabeller, etter kjønn og alder (1966–2025)

### search

In [ ]:
ssb.search("dødelighetstabeller", limit=5)

### describe

In [ ]:
levetid_info = ssb.describe("07902")
levetid_info

### values

Four statistikkvariabler. `ForvGjenLevetid` — forventet gjenstående levetid ved
alder x — is the one this dashboard draws.

In [ ]:
ssb.values("07902", "ContentsCode")

### fetch

`decimals` is reported as 0 for this table, but life expectancy is fractional.
The wrapper checks the actual values rather than trusting the hint, so `value`
comes back `Float64` — another thing that would otherwise be manual cleanup.

In [ ]:
levetid = ssb.fetch(
    "07902",
    Kjonn="*",
    AlderX="*",
    contents="ForvGjenLevetid",
    time="*",
)

storage.save("forventet_gjenstaende_levetid", levetid)
levetid.dtypes

### plot

Forventet gjenstående levetid ved alder 0 — life expectancy at birth — by sex.

In [ ]:
ved_fodsel = levetid[levetid["AlderX_code"] == "000"].sort_values("year")

px.line(
    ved_fodsel,
    x="year",
    y="value",
    color="Kjonn",
    title="Forventet levealder ved fødsel",
    labels={"year": "År", "value": "År", "Kjonn": "Kjønn"},
)

---
## What this notebook produced

In [ ]:
storage.tables()